# 实验六 SmolVLM 网格定位与 RSI
使用实验六独立远端 venv 内核。先读 protocol.json 与 README.md；Notebook 不携带实验输出，已有结果时跳过训练单元，直接查看第 5 节。原始 10 步与授权追加训练分目录保存；默认读取 s60_r30，不混用预算。

In [ ]:
import os, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get('LAB06_ROOT', str(Path.home()/'llm_course_lab06')))
MODEL = Path(os.environ.get('LAB06_MODEL', str(Path.home()/'llm_course_lab05/models/SmolVLM-256M-Instruct')))
PY = ROOT/'.venv/bin/python'
CODE = ROOT/'src/hw6_rsi_localize.py'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
def run(*args, gpu='0'):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
    subprocess.run([str(PY),str(CODE),'--root',str(ROOT),'--model',str(MODEL),*args],env=env,check=True)
print(json.loads((ROOT/'src/protocol.json').read_text()))

## 1 数据与实现核验
55 张图、固定 seed=2026/42；种子 15 题与评测同题。已有数据不要再次生成。model_check 只反传检查，不做优化器更新。

In [ ]:
if not (ROOT/'hw6_imgs/gt.json').exists():
    run('--mode','make_data')
run('--mode','check')
run('--mode','model_check')

## 2 必做：4×4 原始 10 步基准
每阶段 10 步，FP32 + LoRA。此单元会训练；输出目录存在时拒绝覆盖。已有结果直接跳到第 5 节。

In [ ]:
run('--mode','run','--grid','4','--steps','10','--rounds','2',gpu='0')

## 3 选做 A：8×8 网格
使用同一组图、目标、初始化和优化步数；不修改 4×4 输出。

In [ ]:
run('--mode','run','--grid','8','--steps','10','--rounds','2',gpu='1')

## 4 授权追加训练：先学会格式，再验证闭环
原始预算未学会定位，因此另存种子 60 步、两轮各 30 步的 s60_r30，4×4/8×8 各占一张 GPU。此单元会训练，已有结果不要重跑。另有 s60_r10 与 s150_r10 预算对照，详见 README；后者不是新增纠错数据带来的收益。

In [ ]:
subprocess.run(['bash',str(ROOT/'src/run_extended.sh'),'60','30','s60_r30'],env=dict(os.environ,LAB06_ROOT=str(ROOT),LAB06_MODEL=str(MODEL)),check=True)

## 5 结果核验与可视化
只重算派生汇总和图片，不训练；不会修改原始预测或训练日志。LAB06_RUN_NAME 默认为 s60_r30；设为空字符串可查看原始 10 步基准。

In [ ]:
RUN_NAME = os.environ.get('LAB06_RUN_NAME','s60_r30')
RESULTS = ROOT/'outputs'/RUN_NAME
subprocess.run([str(PY),str(ROOT/'src/audit_results.py'),'--root',str(ROOT),'--run-name',RUN_NAME],check=True)
from IPython.display import display, Image
for grid in [4,8]:
    out = RESULTS/f'grid{grid}'
    print((out/'metrics.csv').read_text(encoding='utf-8-sig'))
    print((out/'round_metrics.csv').read_text(encoding='utf-8-sig'))
    display(Image(filename=str(out/'rsi_curves.png')))
    display(Image(filename=str(out/'training_loss.png')))
display(Image(filename=str(RESULTS/'grid_comparison.png')))
if (ROOT/'outputs/extension_budget_comparison.png').exists():
    display(Image(filename=str(ROOT/'outputs/extension_budget_comparison.png')))

## 6 查看原始纠错与数据回流
自查改写不等于改对；工具通过不等于 GT 正确；只有真实生成且通过双重检查的纠错答案才入池。某类代表样例未出现时，不编造。s60_r30 的一条自查纠正被验证、入池并训练；工具反馈单独纠对的正例尚未观察到。种子训练与评测同题，不能作为未见问题泛化成绩。

In [ ]:
out = RESULTS/'grid4'
for phase in ['sft_harness','rsi1_correction','rsi2_correction']:
    records = json.loads((out/phase/'records.json').read_text())
    for r in records:
        if not r['ok1']:
            print(r['id'],r['q'],r['pred1'],r['pred2_self'],r['pred2_tool'],r['final'],r['accepted'])
print(json.loads((out/'parameters.json').read_text()))
probe = ROOT/'outputs'/f'feedback_{RUN_NAME}'/'grid4/feedback_comparison.json'
if probe.exists():
    print(json.loads(probe.read_text()))